In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
!mkdir -p "/content/dataset"
!cp -r "/content/drive/MyDrive/ppe detection.yolov8/"* "/content/dataset/"

In [6]:
!ls "/content/drive/MyDrive/ppe detection.yolov8"

data.yaml  README.roboflow.txt	train


In [7]:
import shutil
import os

# Kaynak ve hedef klasör yolları
kaynak_klasor = "/content/drive/MyDrive/ppe detection.yolov8"
hedef_klasor = "/content/dataset"

# Güvenli kopyalama
if os.path.exists(kaynak_klasor):
    shutil.copytree(kaynak_klasor, hedef_klasor, dirs_exist_ok=True)
    print("✅ Kopyalama işlemi başarıyla tamamlandı!")
    print("Hedef Klasör İçeriği:", os.listdir(hedef_klasor))

    # Train klasörünün içinde ne olduğunu görelim
    train_iceriği = os.path.join(hedef_klasor, "train")
    if os.path.exists(train_iceriği):
        print("Train Klasörü İçeriği:", os.listdir(train_iceriği)[:5], "... (toplam dosya sayısı:", len(os.listdir(train_iceriği)), ")")
else:
    print("❌ Kaynak klasör bulunamadı. Lütfen Drive yolunu kontrol edin.")

✅ Kopyalama işlemi başarıyla tamamlandı!
Hedef Klasör İçeriği: ['README.roboflow.txt', 'train', '.DS_Store', 'data.yaml']
Train Klasörü İçeriği: ['images', 'labels'] ... (toplam dosya sayısı: 2 )


In [8]:
import os
import random
import shutil

base_dir = "/content/dataset"
train_img_dir = os.path.join(base_dir, "train/images")
train_lbl_dir = os.path.join(base_dir, "train/labels")

# Yeni valid ve test klasörlerini oluşturalım
for split in ["valid", "test"]:
    os.makedirs(os.path.join(base_dir, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(base_dir, split, "labels"), exist_ok=True)

# Tüm görselleri listele ve karıştır
all_images = sorted([f for f in os.listdir(train_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
random.seed(42)  # Sonuçlar her seferinde aynı çıksın diye sabitliyoruz
random.shuffle(all_images)

total_files = len(all_images)
num_val = int(total_files * 0.10)
num_test = int(total_files * 0.10)

val_images = all_images[:num_val]
test_images = all_images[num_val:num_val+num_test]

print("Bölme işlemi başlıyor...")

# Validasyon verilerini taşı
for img in val_images:
    lbl = os.path.splitext(img)[0] + ".txt"
    shutil.move(os.path.join(train_img_dir, img), os.path.join(base_dir, "valid/images", img))
    if os.path.exists(os.path.join(train_lbl_dir, lbl)):
        shutil.move(os.path.join(train_lbl_dir, lbl), os.path.join(base_dir, "valid/labels", lbl))

# Test verilerini taşı
for img in test_images:
    lbl = os.path.splitext(img)[0] + ".txt"
    shutil.move(os.path.join(train_img_dir, img), os.path.join(base_dir, "test/images", img))
    if os.path.exists(os.path.join(train_lbl_dir, lbl)):
        shutil.move(os.path.join(train_lbl_dir, lbl), os.path.join(base_dir, "test/labels", lbl))

print("\n✅ Veri seti başarıyla bölündü!")
print("Train Görsel Sayısı:", len(os.listdir(os.path.join(base_dir, "train/images"))))
print("Valid Görsel Sayısı:", len(os.listdir(os.path.join(base_dir, "valid/images"))))
print("Test Görsel Sayısı:", len(os.listdir(os.path.join(base_dir, "test/images"))))

Bölme işlemi başlıyor...

✅ Veri seti başarıyla bölündü!
Train Görsel Sayısı: 943
Valid Görsel Sayısı: 117
Test Görsel Sayısı: 117


In [10]:
import os
import torch
import cv2
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

class PPEYoloDataset(Dataset):
    def __init__(self, image_dir, label_dir, transforms=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        height, width, _ = img.shape

        label_name = os.path.splitext(img_name)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_name)

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        # Faster R-CNN'de 0 her zaman Arka Plandır (Background).
                        # O yüzden sınıflarımızı 1 kaydırıyoruz: kask(0)->1, kafa(1)->2
                        class_id += 1

                        x_c, y_c, w, h = map(float, parts[1:])

                        xmin = (x_c - w / 2) * width
                        ymin = (y_c - h / 2) * height
                        xmax = (x_c + w / 2) * width
                        ymax = (y_c + h / 2) * height

                        if xmax > xmin and ymax > ymin:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(class_id)

        # Boş görsellerde hata almamak için sahte bir kutu ekliyoruz
        if len(boxes) == 0:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        image_id = torch.tensor([idx])
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        img_tensor = torchvision.transforms.functional.to_tensor(img)
        return img_tensor, target

    def __len__(self):
        return len(self.image_files)

def collate_fn(batch):
    return tuple(zip(*batch))

print("✅ Dataset sınıfı başarıyla tanımlandı!")

✅ Dataset sınıfı başarıyla tanımlandı!


In [11]:
train_dataset = PPEYoloDataset("/content/dataset/train/images", "/content/dataset/train/labels")
valid_dataset = PPEYoloDataset("/content/dataset/valid/images", "/content/dataset/valid/labels")

# Batch size'ı T4 GPU'yu zorlamamak için 4 veya 8 yapabiliriz. 4 oldukça güvenlidir.
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print("Eğitim ve Validasyon DataLoader'ları hazır!")

Eğitim ve Validasyon DataLoader'ları hazır!


In [12]:
# Pre-trained Faster R-CNN modelini yükle
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

# Sınıf sayımız: 2 (helmet, head) + 1 (arka plan) = 3
num_classes = 3

# Modelin mevcut sınıflandırıcı kafasının giriş özellik sayısını al
in_features = model.roi_heads.box_predictor.cls_score.in_features

# Yeni sınıflandırıcıyı modele yerleştir
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Modeli GPU'ya taşı
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

print(f"Model başarıyla yüklendi ve {device} cihazına taşındı!")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 152MB/s]


Model başarıyla yüklendi ve cuda cihazına taşındı!


In [13]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 10
print("🚀 Eğitim başlıyor... Her epoch sonrası toplam loss değerini göreceksiniz.")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for images, targets in train_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Faster R-CNN eğitim modundayken loss sözlüğü döner
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    print(f"Epoch [{epoch+1}/{num_epochs}] - Ortalama Loss: {epoch_loss / len(train_loader):.4f}")

print("🎉 Eğitim başarıyla tamamlandı!")
# Eğitilen ağırlıkları kaydetmeyi unutmayalım
torch.save(model.state_dict(), "faster_rcnn_ppe.pth")
print("Model ağırlıkları 'faster_rcnn_ppe.pth' olarak kaydedildi.")

🚀 Eğitim başlıyor... Her epoch sonrası toplam loss değerini göreceksiniz.
Epoch [1/10] - Ortalama Loss: 0.3699
Epoch [2/10] - Ortalama Loss: 0.2648
Epoch [3/10] - Ortalama Loss: 0.2361
Epoch [4/10] - Ortalama Loss: 0.2112
Epoch [5/10] - Ortalama Loss: 0.1911
Epoch [6/10] - Ortalama Loss: 0.1714
Epoch [7/10] - Ortalama Loss: 0.1619
Epoch [8/10] - Ortalama Loss: 0.1475
Epoch [9/10] - Ortalama Loss: 0.1377
Epoch [10/10] - Ortalama Loss: 0.1249
🎉 Eğitim başarıyla tamamlandı!
Model ağırlıkları 'faster_rcnn_ppe.pth' olarak kaydedildi.


In [14]:
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 21.4 MB/s eta 0:00:00


In [15]:
import time
import torch
from torch.utils.data import DataLoader
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Test Verisi İçin DataLoader Hazırlama
test_dataset = PPEYoloDataset("/content/dataset/test/images", "/content/dataset/test/labels")
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print("🔍 Test seti değerlendirmesi başlıyor (mAP ve FPS hesaplanıyor)...")
model.eval() # Modeli test moduna al

# mAP metriğini başlat
metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')

total_time = 0
total_images = 0

with torch.no_grad():
    for images, targets in test_loader:
        images = list(image.to(device) for image in images)

        start_time = time.time()
        outputs = model(images)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end_time = time.time()

        total_time += (end_time - start_time)
        total_images += len(images)

        preds = []
        for out in outputs:
            preds.append({
                "boxes": out["boxes"].cpu(),
                "scores": out["scores"].cpu(),
                "labels": out["labels"].cpu()
            })

        target_list = []
        for t in targets:
            target_list.append({
                "boxes": t["boxes"].cpu(),
                "labels": t["labels"].cpu()
            })

        metric.update(preds, target_list)

# Nihai sonuçları hesapla
results = metric.compute()
fps = total_images / total_time

print("\n" + "="*50)
print("📊 FASTER R-CNN TEST SONUÇLARI")
print("="*50)
print(f"Test Edilen Görsel Sayısı : {total_images}")
print(f"mAP@50 (YOLO karşılığı)   : {results['map_50'].item() * 100:.2f}%")
print(f"Ortalama Hız (FPS)        : {fps:.2f} (YOLO = 103 FPS idi)")
print("="*50)

🔍 Test seti değerlendirmesi başlıyor (mAP ve FPS hesaplanıyor)...

📊 FASTER R-CNN TEST SONUÇLARI
Test Edilen Görsel Sayısı : 117
mAP@50 (YOLO karşılığı)   : 88.73%
Ortalama Hız (FPS)        : 6.42 (YOLO = 103 FPS idi)
